# AI4I 2020 Machine Failure Prediction
Preprocessing, Balancing (SMOTE), Model Training & Evaluation

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')


## Load Data

In [2]:
data = pd.read_csv('C:\\Users\\Ali\\Desktop\\ai4i2020.csv')
data.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Encode Categories & Drop Useless Columns

In [3]:
le = LabelEncoder()
data['Type'] = le.fit_transform(data['Type'])

data = data.drop(columns=['UDI', 'Product ID'])


## Feature Selection

In [4]:
X = data.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = data['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)


##  SMOTE to Fix Class Imbalance

In [5]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print('Before SMOTE:', np.bincount(y_train))
print('After SMOTE:', np.bincount(y_train_res))


Before SMOTE: [7729  271]
After SMOTE: [7729 7729]


##  Scaling

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)


##  Model Training & Evaluation

In [7]:
models = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ('Random Forest', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))
]

for name, model in models:
    print(f"\n===== {name} =====")
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train_res)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    print(classification_report(y_test, y_pred))
    print('ROC-AUC:', roc_auc_score(y_test, y_prob))
    print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))



===== Logistic Regression =====
              precision    recall  f1-score   support

           0       0.99      0.84      0.91      1932
           1       0.15      0.81      0.26        68

    accuracy                           0.84      2000
   macro avg       0.57      0.82      0.58      2000
weighted avg       0.96      0.84      0.89      2000

ROC-AUC: 0.8946611253196931
Confusion Matrix:
 [[1625  307]
 [  13   55]]

===== Random Forest =====
              precision    recall  f1-score   support

           0       0.99      0.96      0.98      1932
           1       0.42      0.74      0.53        68

    accuracy                           0.96      2000
   macro avg       0.70      0.85      0.75      2000
weighted avg       0.97      0.96      0.96      2000

ROC-AUC: 0.957473206673974
Confusion Matrix:
 [[1862   70]
 [  18   50]]
